<a href="https://colab.research.google.com/github/jonathansearch/RATISS-ONE/blob/main/colab/Index_Mini_Cerveau_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Projet RATISS - Micro-Cerveau Sémantique & ScalableGPT (GPU T4)
**RATISS Labs · Auteur : Jonathan Evina (Mono-chercheur, Yaoundé, Cameroun) · Licence MIT**  
**Architecture Tenseur PyTorch GPU · Modèle Génératif ScalableGPT · Double Dataset Simple & Massif**

Ce notebook sert d'environnement clé en main sur votre GPU T4 pour :
1. 📥 Récupérer et exécuter le **Mini-Cerveau Sémantique** (394 µs d'inférence tensorielle).
2. 🚀 Entraîner la **Béquille Générative ScalableGPT** (Transformer décodeur) sur corpus français.
3. 💬 Générer des phrases fluides en sortie.

### 📥 1/3 : Indexation et Téléchargement des Scripts Officiels

In [ ]:
import os, urllib.request

print("📥 Récupération intégrale du dossier mini_cerveau_project...")
base_url = "https://raw.githubusercontent.com/jonathansearch/RATISS-ONE/main/mini_cerveau_project"
fichiers = ["README.md", "mini_cerveau_gpu.py", "scalable_gpt_train.py", "mini_cerveau_gpu.pt"]

os.makedirs("mini_cerveau_project", exist_ok=True)

for f in fichiers:
    url = f"{base_url}/{f}"
    dest = f"mini_cerveau_project/{f}"
    print(f"   • Téléchargement de {f}...")
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp:
        with open(dest, "wb") as out:
            out.write(resp.read())
    print(f"     ✓ {f} reçu ({os.path.getsize(dest)/(1024*1024):.2f} Mo)")

print("
✅ Dossier mini_cerveau_project indexé et prêt !")
!ls -lh mini_cerveau_project

### ⚡ 2/3 : Inférence Haute Vitesse sur le Mini-Cerveau (< 1 ms)

In [ ]:
import sys
sys.path.insert(0, "mini_cerveau_project")
from mini_cerveau_gpu import interroger_mini_cerveau

# Test direct de l'interrogation tensorielle
reponse = interroger_mini_cerveau("Parle moi de situation et de politique", model_path="mini_cerveau_project/mini_cerveau_gpu.pt")
print("🤖 Réponse du Mini-Cerveau :", reponse)

### 🚀 3/3 : Entraînement de la Béquille Générative ScalableGPT (GPU T4)
**Choix du Dataset :**
-  : Corpus littéraire français rapide (Jules Verne) ➔ Idéal pour tester en 2 minutes !
-  : Corpus multi-œuvres à grande échelle (Gutenberg / Misérables / Candide) ➔ Pour un entraînement poussé !

**Choix de l'Échelle :**
-  : 20 Millions de paramètres (ultra-véloce sur GPU T4).
-     : 85 Millions de paramètres (architecture complète officielle).

In [ ]:
# Lancement direct du script d'entraînement écrit à la main
!python mini_cerveau_project/scalable_gpt_train.py --dataset simple --echelle compact --steps 300 --batch-size 16

### 💬 Test de Génération avec la Béquille Entraînée

In [ ]:
import torch
from mini_cerveau_project.scalable_gpt_train import ScalableGPT, TokenizerManager

device = "cuda" if torch.cuda.is_available() else "cpu"
chemin_ckpt = "scalable_gpt_compact.pt"

if os.path.exists(chemin_ckpt):
    ckpt = torch.load(chemin_ckpt, map_location=device)
    cfg = ckpt["config"]
    tok = TokenizerManager()

    model = ScalableGPT(
        vocab_size=cfg["vocab_size"],
        d_model=cfg["d_model"],
        n_head=cfg["n_head"],
        n_layer=cfg["n_layer"],
        block_size=cfg["block_size"]
    ).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    amorce = "Le laboratoire RATISS développe"
    idx_in = torch.tensor([tok.encode(amorce)], device=device)
    out = model.generate(idx_in, max_new_tokens=40, temperature=0.7)
    print("🤖 Texte généré par ScalableGPT :
", tok.decode(out[0].tolist()))
else:
    print("⚠️ Lancez d'abord la cellule d'entraînement ci-dessus pour générer le checkpoint !")